# Feature Engineering — Real Estate Price Prediction (Florida)

Este notebook implementa el pipeline completo de **feature engineering** sobre el dataset de propiedades inmobiliarias de Florida.  
Al final genera archivos `.parquet` (`train_fe`, `meta_train_fe`, `test_fe`) listos para consumir en el script de modelado.

**Estructura:**
1. Imports & Config  
2. Inspección del Dataset  
3. Bins de Precio (estratificación)  
4. Split Estratificado Train / Test  
5. Feature Engineering Functions  
   - 5.1 Size & Area Features  
   - 5.2 Room Features  
   - 5.3 Property Age Features  
   - 5.4 Financial Features  
   - 5.5 Geographic Features  
   - 5.6 School Features  
   - 5.7 Amenity Features  
   - 5.8 Listing Quality Features  
   - 5.9 Text Features  
   - 5.10 Cross / Interaction Features  
   - 5.11 Pipeline Completo — `build_features`

---
## 1. Imports & Config

| Librería | Rol |
|---|---|
| `numpy` / `pandas` | Manipulación de datos |
| `plotly` | Visualizaciones interactivas |
| `sklearn` | Split estratificado |
| `joblib` | Persistencia de modelos |

> **`FLORIDA_ZIP_STATS`**: diccionario con datos demográficos (ACS 2023, US Census Bureau) por código ZIP de Florida — población, área, densidad y renta mediana. Se usará para enriquecer features geográficas.  
> **`process_fe`**: si `True` recalcula el FE y guarda en disco; si `False` carga los parquet pre-existentes.

In [1]:
from __future__ import annotations

import os
import re
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from joblib import dump, load

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    StratifiedGroupKFold,
    train_test_split,
)

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
dir = Path('.')
df_train = pd.read_csv('../../data/tabular/train_processed.csv')
df_test  = pd.read_csv('../../data/tabular/test_processed.csv')


# ── Constants ────────────────────────────────────────────────────────────────
seed       = 42
target     = 'lastSoldPrice_hpi_adjusted'
stratified = ''

# ── Splits ──────────────────────────────────────────────────────────────────
# - Si META_TRAIN_SIZE > 0: split 3-way (train + meta_train + test).
#   Útil si después vas a usar meta_train para entrenar un metamodelo encima.
# - Si META_TRAIN_SIZE = 0: split 2-way (solo train + test). USE_META_TRAIN=False.
TRAIN_SIZE      = 0.8
TEST_SIZE       = 0.2
META_TRAIN_SIZE = 0

USE_META_TRAIN = META_TRAIN_SIZE > 0

# ── Florida ZIP-code demographics (ACS 2023 5-year estimates) ─────────────────
# Source : US Census Bureau, American Community Survey 5-Year Estimates 2023
#          Tables B01003 (population), B19013 (median household income),
#          TIGER/Line ZCTA shapefiles for land area.
# Access : https://data.census.gov  ->  Geographies: ZIP Code Tabulation Areas
#          or via Census API: https://api.census.gov/data/2023/acs/acs5
# Note   : property_age = 2024 - yearBuilt confirms competition data is 2024.
#          pop_density = population / area_km2  (people / km²)
#          median_hhi  = median household income in USD
FLORIDA_ZIP_STATS: dict[int, dict] = {
    # ── Miami-Dade County ──────────────────────────────────────────────────
    33015: {'population':  50_500, 'area_km2':  38, 'pop_density': 1_329, 'median_hhi': 52_000},  # Miami Gardens / NW Miami-Dade
    33018: {'population':  42_000, 'area_km2':  15, 'pop_density': 2_800, 'median_hhi': 48_000},  # Hialeah Gardens
    33032: {'population':  52_000, 'area_km2':  55, 'pop_density':   945, 'median_hhi': 38_000},  # Homestead / Leisure City
    33128: {'population':  14_000, 'area_km2':   3, 'pop_density': 4_667, 'median_hhi': 42_000},  # Downtown Miami
    33129: {'population':  20_000, 'area_km2':   4, 'pop_density': 5_000, 'median_hhi': 88_000},  # Coconut Grove / south Brickell
    33130: {'population':  25_000, 'area_km2':   4, 'pop_density': 6_250, 'median_hhi': 45_000},  # Little Havana / Brickell
    33131: {'population':   9_000, 'area_km2':   2, 'pop_density': 4_500, 'median_hhi': 95_000},  # Brickell / Coconut Grove east
    33132: {'population':  22_000, 'area_km2':   4, 'pop_density': 5_500, 'median_hhi': 55_000},  # Downtown / Edgewater
    33133: {'population':  36_000, 'area_km2':  12, 'pop_density': 3_000, 'median_hhi': 92_000},  # Coconut Grove
    33137: {'population':  20_000, 'area_km2':   6, 'pop_density': 3_333, 'median_hhi': 68_000},  # Design District / MiMo
    33138: {'population':  22_000, 'area_km2':   8, 'pop_density': 2_750, 'median_hhi': 72_000},  # Miami Shores
    33139: {'population':  35_000, 'area_km2':   4, 'pop_density': 8_750, 'median_hhi': 68_000},  # South Beach
    33140: {'population':  18_000, 'area_km2':   3, 'pop_density': 6_000, 'median_hhi': 78_000},  # Mid Beach
    33141: {'population':  35_000, 'area_km2':   5, 'pop_density': 7_000, 'median_hhi': 55_000},  # North Beach
    33143: {'population':  34_000, 'area_km2':  18, 'pop_density': 1_889, 'median_hhi': 95_000},  # South Miami / Pinecrest
    33145: {'population':  42_000, 'area_km2':   5, 'pop_density': 8_400, 'median_hhi': 34_000},  # Little Havana
    33146: {'population':  22_000, 'area_km2':  10, 'pop_density': 2_200, 'median_hhi': 112_000},  # Coral Gables
    33149: {'population':  13_500, 'area_km2':   6, 'pop_density': 2_250, 'median_hhi': 135_000},  # Key Biscayne
    33154: {'population':   6_500, 'area_km2':   2, 'pop_density': 3_250, 'median_hhi': 88_000},  # Bal Harbour / Surfside
    33156: {'population':  42_000, 'area_km2':  28, 'pop_density': 1_500, 'median_hhi': 128_000},  # Pinecrest / Kendall
    33157: {'population':  48_000, 'area_km2':  35, 'pop_density': 1_371, 'median_hhi': 88_000},  # Palmetto Bay / Cutler Bay north
    33158: {'population':  22_000, 'area_km2':  20, 'pop_density': 1_100, 'median_hhi': 108_000},  # Palmetto Bay south
    33160: {'population':  36_000, 'area_km2':   8, 'pop_density': 4_500, 'median_hhi': 52_000},  # North Miami Beach / Aventura
    33161: {'population':  52_000, 'area_km2':  18, 'pop_density': 2_889, 'median_hhi': 44_000},  # North Miami
    33162: {'population':  42_000, 'area_km2':  12, 'pop_density': 3_500, 'median_hhi': 48_000},  # North Miami Beach
    33165: {'population':  50_000, 'area_km2':  15, 'pop_density': 3_333, 'median_hhi': 52_000},  # Westchester / Sweetwater
    33179: {'population':  32_000, 'area_km2':  10, 'pop_density': 3_200, 'median_hhi': 52_000},  # North Miami Beach west
    33180: {'population':  24_000, 'area_km2':   6, 'pop_density': 4_000, 'median_hhi': 65_000},  # Aventura / Country Club
    33181: {'population':  16_000, 'area_km2':   5, 'pop_density': 3_200, 'median_hhi': 48_000},  # North Miami Beach east
    33189: {'population':  32_000, 'area_km2':  22, 'pop_density': 1_455, 'median_hhi': 62_000},  # Cutler Bay
    33190: {'population':   8_000, 'area_km2':  20, 'pop_density':   400, 'median_hhi': 55_000},  # Cutler Bay / southern tip
    # ── Broward County ────────────────────────────────────────────────────
    33009: {'population':  42_000, 'area_km2':  12, 'pop_density': 3_500, 'median_hhi': 52_000},  # Hallandale Beach
    33019: {'population':  35_000, 'area_km2':  12, 'pop_density': 2_917, 'median_hhi': 65_000},  # Hollywood / Dania Beach
    33063: {'population':  52_000, 'area_km2':  22, 'pop_density': 2_364, 'median_hhi': 55_000},  # Pompano Beach / Margate
    33305: {'population':  22_000, 'area_km2':   7, 'pop_density': 3_143, 'median_hhi': 72_000},  # Fort Lauderdale / Wilton Manors
    33308: {'population':  28_000, 'area_km2':  10, 'pop_density': 2_800, 'median_hhi': 62_000},  # Fort Lauderdale / Oakland Park
    33316: {'population':  18_000, 'area_km2':   5, 'pop_density': 3_600, 'median_hhi': 78_000},  # Fort Lauderdale Beach
    # ── Palm Beach County ─────────────────────────────────────────────────
    33401: {'population':  38_000, 'area_km2':  22, 'pop_density': 1_727, 'median_hhi': 42_000},  # West Palm Beach
    33405: {'population':  24_000, 'area_km2':  12, 'pop_density': 2_000, 'median_hhi': 55_000},  # West Palm Beach south
    33431: {'population':  26_000, 'area_km2':  14, 'pop_density': 1_857, 'median_hhi': 92_000},  # Boca Raton west
    33432: {'population':  20_000, 'area_km2':  10, 'pop_density': 2_000, 'median_hhi': 108_000},  # Boca Raton east
    33435: {'population':  35_000, 'area_km2':  18, 'pop_density': 1_944, 'median_hhi': 52_000},  # Boynton Beach
    33441: {'population':  32_000, 'area_km2':  14, 'pop_density': 2_286, 'median_hhi': 56_000},  # Deerfield Beach
    33460: {'population':  38_000, 'area_km2':  16, 'pop_density': 2_375, 'median_hhi': 44_000},  # Lake Worth
    33462: {'population':  22_000, 'area_km2':  12, 'pop_density': 1_833, 'median_hhi': 52_000},  # Lantana / Lake Worth south
    33480: {'population':  10_000, 'area_km2':   8, 'pop_density': 1_250, 'median_hhi': 175_000},  # Palm Beach island
    33483: {'population':  20_000, 'area_km2':  10, 'pop_density': 2_000, 'median_hhi': 88_000},  # Delray Beach east
    33487: {'population':  30_000, 'area_km2':  16, 'pop_density': 1_875, 'median_hhi': 82_000},  # Boca Raton north
    # ── Hillsborough / Tampa area ─────────────────────────────────────────
    33511: {'population':  58_000, 'area_km2':  55, 'pop_density': 1_055, 'median_hhi': 65_000},  # Brandon
    33543: {'population':  42_000, 'area_km2':  62, 'pop_density':   677, 'median_hhi': 88_000},  # Wesley Chapel
    33596: {'population':  32_000, 'area_km2':  42, 'pop_density':   762, 'median_hhi': 72_000},  # Valrico
    33603: {'population':  22_000, 'area_km2':   8, 'pop_density': 2_750, 'median_hhi': 62_000},  # Tampa / Seminole Heights
    # ── Other Florida counties ────────────────────────────────────────────
    32127: {'population':  28_000, 'area_km2':  52, 'pop_density':   538, 'median_hhi': 58_000},  # Port Orange (Volusia)
    32309: {'population':  18_000, 'area_km2':  85, 'pop_density':   212, 'median_hhi': 72_000},  # Tallahassee NE (Leon)
    32444: {'population':  22_000, 'area_km2':  45, 'pop_density':   489, 'median_hhi': 55_000},  # Panama City area (Bay)
    32839: {'population':  32_000, 'area_km2':  28, 'pop_density': 1_143, 'median_hhi': 42_000},  # Orlando south (Orange)
    33870: {'population':  18_000, 'area_km2':  80, 'pop_density':   225, 'median_hhi': 38_000},  # Sebring (Highlands)
    33904: {'population':  28_000, 'area_km2':  18, 'pop_density': 1_556, 'median_hhi': 55_000},  # Cape Coral south (Lee)
    33919: {'population':  35_000, 'area_km2':  38, 'pop_density':   921, 'median_hhi': 52_000},  # Fort Myers SW (Lee)
    34145: {'population':  18_000, 'area_km2':  55, 'pop_density':   327, 'median_hhi': 112_000},  # Marco Island (Collier)
    34205: {'population':  26_000, 'area_km2':  18, 'pop_density': 1_444, 'median_hhi': 50_000},  # Bradenton (Manatee)
    34219: {'population':  18_000, 'area_km2':  95, 'pop_density':   189, 'median_hhi': 68_000},  # Parrish / Ellenton (Manatee)
    34773: {'population':  22_000, 'area_km2': 120, 'pop_density':   183, 'median_hhi': 62_000},  # St. Cloud east (Osceola)
    34990: {'population':  22_000, 'area_km2':  95, 'pop_density':   232, 'median_hhi': 82_000},  # Palm City (Martin)
}

np.random.seed(seed)

# ── Salida del Feature Engineering ───────────────────────────────────────────
# fe_output : directorio donde se guardan/leen los parquet.
# fe_files  : nombre de archivo por partición.
fe_output = '../../data/tabular/'
# fe_files  = {
#     'train'            : 'train_fe.parquet',
#     'meta_train'       : 'meta_train_fe.parquet',
#     'test'             : 'test_fe.parquet',          # 20% split de train_processed (validacion)
#     'competition_test' : 'competition_test_fe.parquet',  # test_processed.csv completo, sin target
# }

fe_files  = {
    'train'            : 'train_fe_sent.parquet',
    'meta_train'       : 'meta_train_fe_sent.parquet',
   'test'             : 'test_fe_sent.parquet',          # 20% split de train_processed (validacion)
    'competition_test' : 'competition_test_fe_sent.parquet',  # test_processed.csv completo, sin target
}


print('Config OK')

Config OK


---
## 2. Inspección del Dataset

Verificamos dimensiones y nombres de columnas del CSV de entrenamiento procesado para confirmar que la carga fue correcta antes de continuar.


In [2]:

print(f'Shape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')
df_train.head(3)

Shape: (11840, 46)
Columns: ['zpid', 'lastSoldPrice_hpi_adjusted', 'log_price', 'description', 'bedrooms', 'bathrooms', 'livingArea', 'yearBuilt', 'latitude', 'longitude', 'lotAreaValue', 'photoCount', 'taxAssessedValue', 'propertyTaxRate', 'homeType', 'zipcode', 'latest_tax_value', 'latest_tax_paid', 'num_tax_records', 'num_sales', 'num_price_changes', 'last_listing_price', 'avg_school_rating', 'max_school_rating', 'num_nearby_schools', 'min_school_distance', 'has_hoa', 'hoa_fee_monthly', 'has_pool', 'has_garage', 'has_waterfront', 'tag_price_cut', 'tag_new_construction', 'tag_foreclosure', 'property_age', 'bath_to_bed_ratio', 'log_living_area', 'log_lot_area', 'zip_3digit', 'desc_length', 'desc_word_count', 'desc_is_boilerplate', 'desc_mentions_renovated', 'desc_mentions_pool', 'desc_mentions_view', 'desc_mentions_new']


,zpid,lastSoldPrice_hpi_adjusted,log_price,description,bedrooms,bathrooms,livingArea,yearBuilt,latitude,longitude,...,log_living_area,log_lot_area,zip_3digit,desc_length,desc_word_count,desc_is_boilerplate,desc_mentions_renovated,desc_mentions_pool,desc_mentions_view,desc_mentions_new
0,1011541,255691.210988,12.451730,This 665 square foot condo home has 1 bedrooms...,1.0,1.0,665.0,2006.0,26.616192,-80.05406,...,6.501290,0.693147,334,136,26,1,0,0,0,0
1,1005445,687261.362622,13.440471,This 1030 square foot condo home has 2 bedroom...,2.0,2.0,1030.0,1987.0,25.847658,-80.14571,...,6.938284,NaN,331,148,28,1,0,0,0,0
2,1002822,264917.298188,12.487177,This 980 square foot condo home has 2 bedrooms...,2.0,2.0,980.0,1974.0,26.307700,-80.09503,...,6.888572,NaN,334,140,27,1,0,0,0,0


---
## 3. Bins de Precio

> **Regla de Sturges**: `N_BINS = ceil(log2(n) + 1)`, acotado entre 4 y 10. Garantiza una granularidad proporcional al tamaño del dataset sin crear demasiadas clases minoritarias.

Se crean **quantile bins** uniformes sobre el target (`lastSoldPrice_hpi_adjusted`) que se usarán como estrato en el split posterior. La distribución resultante se visualiza con límites de corte superpuestos.

| Parámetro | Descripción |
|---|---|
| `N_BINS` | Número de bins calculado por la regla de Sturges |
| `price_bin` | Columna nueva con etiquetas Q1…Q10 |
| `edges` | Límites de cada bin (para visualización y referencia) |


In [3]:
# ── Número de bins: regla de Sturges, acotado entre 4 y 10 ───────────────────
n = len(df_train[target].dropna())
N_BINS = int(np.clip(np.ceil(np.log2(n) + 1), 4, 10))
print(f'N_BINS = {N_BINS}  (Sturges, n={n:,})')

bin_labels = [f'Q{i+1}' for i in range(N_BINS)]
df_train['price_bin'], edges = pd.qcut(
    df_train[target], q=N_BINS, labels=bin_labels,
    duplicates='drop', retbins=True,
)

dist = df_train['price_bin'].value_counts().sort_index()
for lbl, cnt in dist.items():
    idx = list(bin_labels).index(lbl)
    print(f'  {lbl}  [{edges[idx]:>9,.0f} – {edges[idx+1]:>9,.0f}]  n={cnt:,}  ({cnt/len(df_train)*100:.1f}%)')

# Distribución con límites de bins
fig = px.histogram(df_train, x=target, nbins=80,
                   title=f'Distribución del target — límites de bins (N={N_BINS}, Sturges)')
for edge in edges[1:-1]:
    fig.add_vline(x=edge, line_dash='dash', line_color='red')
fig.update_layout(height=380, showlegend=False)
fig.show()

N_BINS = 10  (Sturges, n=11,840)
  Q1  [   51,276 –   186,923]  n=1,184  (10.0%)
  Q2  [  186,923 –   269,069]  n=1,184  (10.0%)
  Q3  [  269,069 –   339,988]  n=1,184  (10.0%)
  Q4  [  339,988 –   407,390]  n=1,184  (10.0%)
  Q5  [  407,390 –   472,289]  n=1,184  (10.0%)
  Q6  [  472,289 –   550,826]  n=1,184  (10.0%)
  Q7  [  550,826 –   656,126]  n=1,184  (10.0%)
  Q8  [  656,126 –   813,868]  n=1,184  (10.0%)
  Q9  [  813,868 – 1,076,245]  n=1,184  (10.0%)
  Q10  [1,076,245 – 1,986,843]  n=1,184  (10.0%)


---
## 4. Split Estratificado Train / Test

> **Estrategia de split**: al estratificar por `price_bin` se garantiza que cada partición tiene la misma distribución de precios que el conjunto original (10 % por cuantil). Esto evita sesgo de selección en la evaluación.

Soporta dos modos controlados por `USE_META_TRAIN`:

| Modo | Particiones | Cuándo usarlo |
|---|---|---|
| `META_TRAIN_SIZE = 0` | `train` + `test` (2-way) | Pipeline estándar |
| `META_TRAIN_SIZE > 0` | `train` + `meta_train` + `test` (3-way) | Entrenamiento de metamodelos (stacking) |


In [4]:
# ── Separación estratificada ──────────────────────────────────────────────────
if USE_META_TRAIN:
    df_train, temp = train_test_split(
        df_train, train_size=TRAIN_SIZE, random_state=seed,
        stratify=df_train['price_bin'],
    )
    relative_test = TEST_SIZE / (TEST_SIZE + META_TRAIN_SIZE)
    df_meta_train, df_meta_test = train_test_split(
        temp, test_size=relative_test, random_state=seed,
        stratify=temp['price_bin'],
    )
else:
    df_train, df_meta_test = train_test_split(
        df_train, train_size=TRAIN_SIZE, test_size=TEST_SIZE,
        random_state=seed, stratify=df_train['price_bin'],
    )
    df_meta_train = None

print(f'Train: {df_train.shape}  |  Test: {df_meta_test.shape}' +
      (f'  |  Meta train: {df_meta_train.shape}' if USE_META_TRAIN else ''))

# ── Reportes ───────────────────────────────────────────────────────────────
print(f'Modo split: {"3-way (train + meta_train + test)" if USE_META_TRAIN else "2-way (train + test)"}')
print(f'Train:      {df_train.shape}')
if USE_META_TRAIN:
    print(f'Meta train: {df_meta_train.shape}')
print(f'Test:       {df_meta_test.shape}')
print('\nDistribución del target en train:')
print((df_train['price_bin'].value_counts(normalize=True) * 100).sort_index().round(1).to_string())

if USE_META_TRAIN:
    print('\nDistribución del target en meta_train:')
    print((df_meta_train['price_bin'].value_counts(normalize=True) * 100).sort_index().round(1).to_string())

print('\nDistribución del target en test:')
print((df_meta_test['price_bin'].value_counts(normalize=True) * 100).sort_index().round(1).to_string())


Train: (9472, 47)  |  Test: (2368, 47)
Modo split: 2-way (train + test)
Train:      (9472, 47)
Test:       (2368, 47)

Distribución del target en train:
price_bin
Q1     10.0
Q2     10.0
Q3     10.0
Q4     10.0
Q5     10.0
Q6     10.0
Q7     10.0
Q8     10.0
Q9     10.0
Q10    10.0

Distribución del target en test:
price_bin
Q1     10.0
Q2     10.0
Q3     10.0
Q4     10.0
Q5     10.0
Q6     10.0
Q7     10.0
Q8     10.0
Q9     10.0
Q10    10.0


---
## 5. Feature Engineering Functions

### 5.1 Size & Area Features

In [5]:
def add_size_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de superficie y proporciones de espacio habitable.

    Estadísticas aprendidas de train: percentil 99 de livingArea para clip,
    percentiles 25 y 75 para los flags de unidad pequeña/grande.

    Nuevas columnas:
        living_area_clipped, total_rooms,
        living_area_per_bed, living_area_per_bath, living_area_per_room,
        lot_to_living_ratio, log_lot_to_living,
        is_small_unit, is_large_unit.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    # Percentiles aprendidos SOLO en train
    area_p25 = float(df_train['livingArea'].quantile(0.25))
    area_p75 = float(df_train['livingArea'].quantile(0.75))
    area_p99 = float(df_train['livingArea'].quantile(0.99))

    for df in [df_train, df_test]:
        df['living_area_clipped']  = df['livingArea'].clip(upper=area_p99)

        beds  = df['bedrooms'].fillna(0).clip(lower=1)
        baths = df['bathrooms'].fillna(0).clip(lower=1)
        df['total_rooms']          = df['bedrooms'].fillna(0) + df['bathrooms'].fillna(0)

        # Espacio por tipo de habitación
        df['living_area_per_bed']  = df['livingArea'] / beds
        df['living_area_per_bath'] = df['livingArea'] / baths
        df['living_area_per_room'] = df['livingArea'] / df['total_rooms'].clip(lower=1)

        # Relación lote / área habitable (0 si lote es nulo)
        lot = df['lotAreaValue'].fillna(0).clip(lower=0)
        df['lot_to_living_ratio']  = lot / df['livingArea'].clip(lower=1)
        df['log_lot_to_living']    = np.log1p(df['lot_to_living_ratio'])

        # Flags de tamaño basados en cuartiles del train
        df['is_small_unit']        = (df['livingArea'] < area_p25).astype(int)
        df['is_large_unit']        = (df['livingArea'] > area_p75).astype(int)

    return df_train, df_test

### 5.2 Room Features

In [6]:
def add_room_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de habitaciones y baños.

    Operaciones aritméticas puras (sin estadísticas de train).

    Nuevas columnas:
        bed_bath_sum, excess_bath, is_studio,
        has_many_beds, has_many_baths, is_bachelor.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    for df in [df_train, df_test]:
        beds  = df['bedrooms'].fillna(0)
        baths = df['bathrooms'].fillna(0)

        df['bed_bath_sum']   = beds + baths

        # Baños en exceso respecto a cuartos (indicador de lujo)
        df['excess_bath']    = (baths - beds).clip(lower=0)

        df['is_studio']      = (beds == 0).astype(int)
        df['has_many_beds']  = (beds >= 4).astype(int)
        df['has_many_baths'] = (baths >= 3).astype(int)

        # Unidad compacta: 1 cuarto + máximo 1 baño (perfil soltero/pareja)
        df['is_bachelor']    = ((beds == 1) & (baths <= 1)).astype(int)

    return df_train, df_test

### 5.3 Property Age Features

In [7]:
def add_property_age_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de antigüedad y era de construcción de la propiedad.

    Estadísticas aprendidas de train: mediana de yearBuilt/property_age
    para imputación, percentil 95 de property_age para clip.

    Nuevas columnas:
        decade, era, is_new_build, is_mid_century, is_vintage,
        age_clipped, log_age, age_squared.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    year_median = float(df_train['yearBuilt'].median())
    age_median  = float(df_train['property_age'].median())
    age_p95     = float(df_train['property_age'].quantile(0.95))

    # Eras como etiquetas numéricas (compatibles con LightGBM sin encoding extra)
    era_bins   = [-np.inf, 10.0, 25.0, 45.0, 65.0, np.inf]
    era_labels = [0, 1, 2, 3, 4]   # muy_nueva / nueva / madura / antigua / vintage

    for df in [df_train, df_test]:
        year = df['yearBuilt'].fillna(year_median)
        age  = df['property_age'].fillna(age_median)

        df['decade']        = ((year // 10) * 10).astype('Int64')

        df['era']           = pd.cut(
            age, bins=era_bins, labels=era_labels,
        ).astype('Int64').astype(float)

        # Flags de época de construcción (umbrales fijos, sin target)
        df['is_new_build']    = (age <= 5).astype(int)
        df['is_mid_century']  = ((year >= 1950) & (year < 1980)).astype(int)
        df['is_vintage']      = (year < 1950).astype(int)

        # Transformaciones de antigüedad para capturar no-linealidad
        df['age_clipped']   = age.clip(upper=age_p95)
        df['log_age']       = np.log1p(df['age_clipped'])
        df['age_squared']   = df['age_clipped'] ** 2

    return df_train, df_test

### 5.4 Financial Features

In [8]:
def add_financial_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features fiscales y financieros derivados de las columnas de tasación
    y precio de listado.

    Estadísticas aprendidas de train: medianas para imputación de nulos,
    percentil 75 de HOA para el flag has_high_hoa.

    Nota: taxAssessedValue y last_listing_price son features disponibles en
    el test set — usarlos NO genera leakage. El target NO se usa en ningún
    cálculo de esta función.

    Nuevas columnas:
        log_tax_assessed, log_listing_price_fe,
        listing_minus_tax, log_listing_minus_tax, tax_to_listing_ratio,
        tax_per_sqft, listing_per_sqft,
        hoa_annual, log_hoa_fee, has_high_hoa,
        effective_tax_rate_implied.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    # Estadísticas aprendidas solo de train
    tax_median     = float(df_train['taxAssessedValue'].median())
    listing_median = float(df_train['last_listing_price'].median())
    hoa_p75        = float(
        df_train.loc[df_train['has_hoa'] == 1, 'hoa_fee_monthly'].quantile(0.75)
    )
    tax_rate_med   = float(df_train['propertyTaxRate'].median())

    for df in [df_train, df_test]:
        tax     = df['taxAssessedValue'].fillna(tax_median)
        listing = df['last_listing_price'].fillna(listing_median)
        area    = df['livingArea'].clip(lower=1)
        hoa     = df['hoa_fee_monthly'].fillna(0)

        df['log_tax_assessed']      = np.log1p(tax)
        df['log_listing_price_fe']  = np.log1p(listing)

        # Diferencia y ratio entre tasación catastral y precio de listado
        gap = listing - tax
        df['listing_minus_tax']     = gap
        df['log_listing_minus_tax'] = np.log1p(np.abs(gap)) * np.sign(gap)
        df['tax_to_listing_ratio']  = tax / listing.clip(lower=1)

        # Precio por pie cuadrado (dos perspectivas)
        df['tax_per_sqft']          = tax / area
        df['listing_per_sqft']      = listing / area

        # HOA
        df['hoa_annual']            = hoa * 12
        df['log_hoa_fee']           = np.log1p(hoa)
        df['has_high_hoa']          = (hoa > hoa_p75).astype(int)

        # Tasa efectiva de impuesto implícita: impuesto pagado / valor catastral
        tax_paid = df['latest_tax_paid'].fillna(
            tax * df['propertyTaxRate'].fillna(tax_rate_med)
        )
        df['effective_tax_rate_implied'] = (tax_paid / tax.clip(lower=1)).clip(upper=0.05)

    return df_train, df_test

### 5.5 Geographic Features

In [9]:
def add_geographic_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features geográficos: frequency encoding de ZIP, estadísticas censales
    del diccionario FLORIDA_ZIP_STATS y binning de coordenadas.

    Estadísticas aprendidas de train: frecuencias de zipcode y zip_3digit.
    Datos estáticos: FLORIDA_ZIP_STATS (fuente ACS 2023, sin leakage).

    Nuevas columnas:
        zip_freq, zip_3digit_freq,
        zip_population, zip_area_km2, zip_pop_density, zip_median_hhi,
        dist_to_miami_center, lat_bin, lon_bin.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    # Frequency encoding — se aprende SOLO en train
    zip_freq_map  = df_train['zipcode'].astype(str).value_counts().to_dict()
    zip3_freq_map = df_train['zip_3digit'].astype(str).value_counts().to_dict()

    # Mapas desde FLORIDA_ZIP_STATS (hardcoded, sin leakage)
    pop_map     = {k: v['population']  for k, v in FLORIDA_ZIP_STATS.items()}
    area_map    = {k: v['area_km2']    for k, v in FLORIDA_ZIP_STATS.items()}
    density_map = {k: v['pop_density'] for k, v in FLORIDA_ZIP_STATS.items()}
    hhi_map     = {k: v['median_hhi']  for k, v in FLORIDA_ZIP_STATS.items()}

    med_pop     = float(np.median(list(pop_map.values())))
    med_area    = float(np.median(list(area_map.values())))
    med_density = float(np.median(list(density_map.values())))
    med_hhi     = float(np.median(list(hhi_map.values())))

    MIAMI_LAT, MIAMI_LON = 25.7617, -80.1918

    for df in [df_train, df_test]:
        # Frequency encoding
        df['zip_freq']        = df['zipcode'].astype(str).map(zip_freq_map).fillna(1.0)
        df['zip_3digit_freq'] = df['zip_3digit'].astype(str).map(zip3_freq_map).fillna(1.0)

        # Estadísticas censales — lookup por ZIP (entero)
        zipc_int = df['zipcode'].astype(str).str.strip().astype(float).astype('Int64')
        df['zip_population']  = zipc_int.map(pop_map).fillna(med_pop)
        df['zip_area_km2']    = zipc_int.map(area_map).fillna(med_area)
        df['zip_pop_density'] = zipc_int.map(density_map).fillna(med_density)
        df['zip_median_hhi']  = zipc_int.map(hhi_map).fillna(med_hhi)

        # Distancia euclidiana al centro de Miami (lat/lon en grados)
        lat = df['latitude'].fillna(MIAMI_LAT)
        lon = df['longitude'].fillna(MIAMI_LON)
        df['dist_to_miami_center'] = np.sqrt(
            (lat - MIAMI_LAT) ** 2 + (lon - MIAMI_LON) ** 2
        )

        # Grid geográfico de 0.05° (~5 km): bins discretos para lat/lon
        df['lat_bin'] = (lat / 0.05).round().astype(int)
        df['lon_bin'] = (lon / 0.05).round().astype(int)

    return df_train, df_test

### 5.6 School Features

In [10]:
def add_school_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de calidad educativa del entorno.

    Estadísticas aprendidas de train: mediana de ratings para imputación,
    percentil 75 de avg_school_rating para el flag has_good_school.

    Nuevas columnas:
        school_quality_gap, schools_per_mile,
        has_top_school, has_good_school, school_rating_tier.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    avg_median  = float(df_train['avg_school_rating'].median())
    maxr_median = float(df_train['max_school_rating'].median())
    dist_median = float(df_train['min_school_distance'].median())
    rating_p75  = float(df_train['avg_school_rating'].quantile(0.75))

    for df in [df_train, df_test]:
        avg  = df['avg_school_rating'].fillna(avg_median)
        maxr = df['max_school_rating'].fillna(maxr_median)
        dist = df['min_school_distance'].fillna(dist_median)
        n    = df['num_nearby_schools'].fillna(0)

        # Dispersión interna de calidad educativa
        df['school_quality_gap'] = (maxr - avg).clip(lower=0)

        # Densidad de oferta: escuelas accesibles por milla de distancia
        df['schools_per_mile']   = n / (dist + 0.1)

        df['has_top_school']     = (maxr >= 8).astype(int)
        df['has_good_school']    = (avg >= rating_p75).astype(int)

        # Tier numérico 0–4 (bins fijos, sin depender del target)
        df['school_rating_tier'] = pd.cut(
            avg,
            bins=[-np.inf, 2.0, 4.0, 6.0, 8.0, np.inf],
            labels=[0, 1, 2, 3, 4],
        ).astype('Int64').astype(float)

    return df_train, df_test

### 5.7 Amenity Features

In [11]:
# Compiled regexes for amenity text signals (defined at module level for reuse)
_RE_AMENITY_GARAGE      = re.compile(r'\b(garage|carport|parking\s+space)\b', re.I)
_RE_AMENITY_WATERFRONT  = re.compile(r'\b(waterfront|oceanfront|bayfront|lakefront|canalfront|intracoastal)\b', re.I)
_RE_AMENITY_HOA         = re.compile(r'\b(HOA|homeowners?\s+association)\b', re.I)
_RE_AMENITY_FORECLOSURE = re.compile(r'\b(foreclosure|bank[\s\-]?owned|REO|short\s+sale)\b', re.I)
_RE_AMENITY_PRICE_CUT   = re.compile(r'\b(price\s+reduced|price\s+cut|price\s+drop|reduced\s+price|below\s+market)\b', re.I)
_RE_AMENITY_NEW_CONSTR  = re.compile(r'\b(new\s+construction|newly\s+(?:built|constructed)|brand[\s\-]new\s+home|new\s+build)\b', re.I)


def add_amenity_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de amenidades combinando campos estructurados con señales de texto.

    Para cada flag booleano estructurado se genera una señal de texto
    (desc_mentions_*) y un indicador combinado (*_any = structured OR text).

    Señales de texto del dataset original (pre-calculadas):
        desc_mentions_pool  ← ya existe en train/test_processed.csv

    Señales de texto creadas aquí via regex sobre 'description':
        desc_mentions_garage, desc_mentions_waterfront, desc_mentions_hoa,
        desc_mentions_foreclosure, desc_mentions_price_cut,
        desc_mentions_new_construction

    Indicadores combinados (structured OR text):
        pool_any, garage_any, waterfront_any, hoa_any,
        foreclosure_any, price_cut_any, new_construction_any

    Features derivados (usan señales combinadas):
        premium_amenities_count, has_waterfront_pool,
        is_distressed, is_luxury_tagged,
        hoa_pool_combo, total_boolean_flags.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    for df in [df_train, df_test]:
        text = df['description'].fillna('')

        # ── Señales estructuradas ─────────────────────────────────────────
        pool        = df['has_pool'].fillna(0).astype(int)
        garage      = df['has_garage'].fillna(0).astype(int)
        waterfront  = df['has_waterfront'].fillna(0).astype(int)
        hoa         = df['has_hoa'].fillna(0).astype(int)
        foreclosure = df['tag_foreclosure'].fillna(0).astype(int)
        price_cut   = df['tag_price_cut'].fillna(0).astype(int)
        new_constr  = df['tag_new_construction'].fillna(0).astype(int)

        # ── Señales de texto ──────────────────────────────────────────────
        # pool: ya pre-calculado en el dataset
        desc_pool        = df['desc_mentions_pool'].fillna(0).astype(int)
        # resto: extraídos aquí via regex
        desc_garage      = text.str.contains(_RE_AMENITY_GARAGE,      na=False).astype(int)
        desc_waterfront  = text.str.contains(_RE_AMENITY_WATERFRONT,  na=False).astype(int)
        desc_hoa         = text.str.contains(_RE_AMENITY_HOA,         na=False).astype(int)
        desc_foreclosure = text.str.contains(_RE_AMENITY_FORECLOSURE, na=False).astype(int)
        desc_price_cut   = text.str.contains(_RE_AMENITY_PRICE_CUT,   na=False).astype(int)
        desc_new_constr  = text.str.contains(_RE_AMENITY_NEW_CONSTR,  na=False).astype(int)

        df['desc_mentions_garage']           = desc_garage
        df['desc_mentions_waterfront']       = desc_waterfront
        df['desc_mentions_hoa']              = desc_hoa
        df['desc_mentions_foreclosure']      = desc_foreclosure
        df['desc_mentions_price_cut']        = desc_price_cut
        df['desc_mentions_new_construction'] = desc_new_constr

        # ── Indicadores combinados (structured OR text) ───────────────────
        pool_any        = (pool        | desc_pool       ).astype(int)
        garage_any      = (garage      | desc_garage     ).astype(int)
        waterfront_any  = (waterfront  | desc_waterfront ).astype(int)
        hoa_any         = (hoa         | desc_hoa        ).astype(int)
        foreclosure_any = (foreclosure | desc_foreclosure).astype(int)
        price_cut_any   = (price_cut   | desc_price_cut  ).astype(int)
        new_constr_any  = (new_constr  | desc_new_constr ).astype(int)

        df['pool_any']             = pool_any
        df['garage_any']           = garage_any
        df['waterfront_any']       = waterfront_any
        df['hoa_any']              = hoa_any
        df['foreclosure_any']      = foreclosure_any
        df['price_cut_any']        = price_cut_any
        df['new_construction_any'] = new_constr_any

        # ── Features compuestos ───────────────────────────────────────────
        # Score de amenidades premium (0–3)
        df['premium_amenities_count'] = pool_any + garage_any + waterfront_any

        # Frente al agua CON piscina
        df['has_waterfront_pool']     = (waterfront_any & pool_any).astype(int)

        # Señales de distress
        df['is_distressed']           = (foreclosure_any | price_cut_any).astype(int)

        # Lujo: frente al agua o piscina+garage
        df['is_luxury_tagged']        = (waterfront_any | (pool_any & garage_any)).astype(int)

        # HOA + piscina (condos premium con mantenimiento)
        df['hoa_pool_combo']          = (hoa_any & pool_any).astype(int)

        # Total de flags booleanos combinados
        df['total_boolean_flags']     = (
            pool_any + garage_any + waterfront_any + hoa_any
            + foreclosure_any + price_cut_any + new_constr_any
        )

    return df_train, df_test

### 5.8 Listing Quality Features

In [12]:
def add_listing_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de calidad y actividad del listado.

    Estadísticas aprendidas de train: percentiles de photoCount para
    los flags de baja/alta cobertura fotográfica, y bins de quintiles.

    Nuevas columnas:
        log_photo_count, photo_quintile, is_low_photo, is_high_photo,
        has_price_history, listing_engagement_score.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    photo_p20 = float(df_train['photoCount'].quantile(0.20))
    photo_p80 = float(df_train['photoCount'].quantile(0.80))

    # Bins de quintiles aprendidos de train, aplicados a test con pd.cut
    _, photo_bins = pd.qcut(
        df_train['photoCount'], q=5, retbins=True, duplicates='drop',
    )

    for df in [df_train, df_test]:
        photos  = df['photoCount'].fillna(0)
        n_sales = df['num_sales'].fillna(0)
        n_chg   = df['num_price_changes'].fillna(0)

        df['log_photo_count']  = np.log1p(photos)
        df['is_low_photo']     = (photos <= photo_p20).astype(int)
        df['is_high_photo']    = (photos >= photo_p80).astype(int)

        df['photo_quintile']   = pd.cut(
            photos, bins=photo_bins, labels=False, include_lowest=True,
        ).fillna(0).astype(int)

        df['has_price_history']         = (n_chg > 0).astype(int)
        df['listing_engagement_score']  = (
            np.log1p(photos) + np.log1p(n_sales) + np.log1p(n_chg)
        )

    return df_train, df_test

### 5.9 Text Features

In [13]:
def add_text_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features adicionales derivados del texto de la descripción.

    Estadísticas aprendidas de train: umbral de descripción detallada
    (percentil 75 de desc_word_count).
    Los regex se aplican sobre la columna 'description' sin usar el target.

    Requiere que add_amenity_features haya corrido antes (crea
    desc_mentions_garage, desc_mentions_waterfront, desc_mentions_hoa,
    desc_mentions_foreclosure, desc_mentions_price_cut,
    desc_mentions_new_construction).

    Nuevas columnas:
        desc_mentions_count, desc_info_density, is_detailed_desc,
        desc_mentions_luxury, desc_mentions_updated,
        desc_mentions_waterfront_kw, desc_mentions_garage_kw,
        desc_has_numbers.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    word_p75 = float(df_train['desc_word_count'].quantile(0.75))

    # Incluye las menciones pre-calculadas del dataset original
    # y las creadas en add_amenity_features (paso 7)
    mention_cols = [
        'desc_mentions_renovated', 'desc_mentions_pool',
        'desc_mentions_view',      'desc_mentions_new',
        'desc_mentions_garage',    'desc_mentions_waterfront',
        'desc_mentions_hoa',       'desc_mentions_foreclosure',
        'desc_mentions_price_cut', 'desc_mentions_new_construction',
    ]

    re_luxury     = re.compile(r'\b(luxury|resort|exclusive|premium|upscale|elegant)\b', re.I)
    re_updated    = re.compile(r'\b(updated|upgraded|remodeled|renovated|modern)\b',     re.I)
    re_waterfront = re.compile(r'\b(waterfront|ocean|bay|lake|canal|water\s*view)\b',    re.I)
    re_garage     = re.compile(r'\b(garage|parking|carport)\b',                          re.I)
    re_numbers    = re.compile(r'\d+')

    for df in [df_train, df_test]:
        text = df['description'].fillna('')

        # Suma de todas las menciones de amenidades (dataset + amenity features)
        df['desc_mentions_count']       = df[mention_cols].fillna(0).sum(axis=1)

        # Compactitud: palabras por carácter
        df['desc_info_density']         = df['desc_word_count'] / (df['desc_length'] + 1)

        # Descripción detallada respecto al umbral del train
        df['is_detailed_desc']          = (df['desc_word_count'] > word_p75).astype(int)

        # Keywords adicionales (patrones más amplios que los de amenity features)
        df['desc_mentions_luxury']        = text.str.contains(re_luxury,     na=False).astype(int)
        df['desc_mentions_updated']       = text.str.contains(re_updated,    na=False).astype(int)
        df['desc_mentions_waterfront_kw'] = text.str.contains(re_waterfront, na=False).astype(int)
        df['desc_mentions_garage_kw']     = text.str.contains(re_garage,     na=False).astype(int)
        df['desc_has_numbers']            = text.str.contains(re_numbers,    na=False).astype(int)

    return df_train, df_test

### 5.10 Cross / Interaction Features

In [14]:
def add_cross_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de interacción entre grupos (aritmética pura, sin leakage).

    Requiere que los siguientes grupos hayan corrido antes:
        add_size_features         → total_rooms
        add_property_age_features → age_clipped
        add_financial_features    → log_listing_price_fe, log_tax_assessed
        add_geographic_features   → zip_median_hhi, zip_pop_density
        add_listing_features      → log_photo_count
        add_amenity_features      → pool_any, waterfront_any

    Nuevas columnas:
        area_x_baths, area_x_waterfront, area_x_pool,
        age_x_area, school_x_area, hhi_x_school,
        density_x_area, tax_per_room, foreclosure_x_age,
        new_constr_x_area, photo_x_listing_price, hoa_x_area.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    for df in [df_train, df_test]:
        area   = df['livingArea'].clip(lower=1)
        age    = df['age_clipped']
        school = df['avg_school_rating'].fillna(0)
        rooms  = df['total_rooms'].clip(lower=1)

        # Área × atributos de calidad (usa señales combinadas structured+text)
        df['area_x_baths']      = area * df['bathrooms'].fillna(0)
        df['area_x_waterfront'] = area * df['waterfront_any']
        df['area_x_pool']       = area * df['pool_any']

        # Antigüedad × tamaño (grande nuevo vs. grande viejo)
        df['age_x_area']        = age * area

        # Escuelas × área (location quality × size)
        df['school_x_area']     = school * area

        # Renta mediana ZIP × rating escuelas (proxy de vecindario de alto valor)
        df['hhi_x_school']      = df['zip_median_hhi'] * school

        # Densidad poblacional × área (urbano denso × tamaño)
        df['density_x_area']    = df['zip_pop_density'] * area

        # Impuesto catastral por habitación
        df['tax_per_room']      = df['taxAssessedValue'].fillna(0) / rooms

        # Distress × antigüedad
        df['foreclosure_x_age'] = df['foreclosure_any'] * age

        # Nueva construcción × área (premium de m² en obra nueva)
        df['new_constr_x_area'] = df['new_construction_any'] * area

        # Cobertura fotográfica × precio de listado
        df['photo_x_listing_price'] = df['log_photo_count'] * df['log_listing_price_fe']

        # HOA fee por pie cuadrado (costo de mantenimiento relativo al tamaño)
        df['hoa_x_area']        = df['hoa_fee_monthly'].fillna(0) / area

    return df_train, df_test

### 5.11 Pipeline Completo — `build_features`

In [15]:
def build_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pipeline completo de Feature Engineering.

    Aplica 10 grupos de features en orden de dependencia. Todas las
    estadísticas se aprenden exclusivamente de df_train y se aplican
    a df_test — sin leakage del target.

    Grupos:
        1.  size      — áreas y proporciones de superficie
        2.  rooms     — habitaciones y baños
        3.  age       — antigüedad y era de construcción
        4.  financial — features fiscales y financieros
        5.  geo       — ZIP encoding, estadísticas censales, coordenadas
        6.  schools   — calidad educativa del entorno
        7.  amenities — amenidades y combinaciones booleanas
        8.  listing   — calidad del listado (fotos, historial)
        9.  text      — keywords y métricas de la descripción
        10. cross     — interacciones entre grupos anteriores

    Returns:
        (df_train_fe, df_test_fe) con todas las nuevas columnas añadidas.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()

    print('FE 1/10 — size features...')
    df_train, df_test = add_size_features(df_train, df_test)

    print('FE 2/10 — room features...')
    df_train, df_test = add_room_features(df_train, df_test)

    print('FE 3/10 — property age features...')
    df_train, df_test = add_property_age_features(df_train, df_test)

    print('FE 4/10 — financial features...')
    df_train, df_test = add_financial_features(df_train, df_test)

    print('FE 5/10 — geographic features...')
    df_train, df_test = add_geographic_features(df_train, df_test)

    print('FE 6/10 — school features...')
    df_train, df_test = add_school_features(df_train, df_test)

    print('FE 7/10 — amenity features...')
    df_train, df_test = add_amenity_features(df_train, df_test)

    print('FE 8/10 — listing features...')
    df_train, df_test = add_listing_features(df_train, df_test)

    print('FE 9/10 — text features...')
    df_train, df_test = add_text_features(df_train, df_test)

    print('FE 10/10 — cross/interaction features...')
    df_train, df_test = add_cross_features(df_train, df_test)

    print(f'FE completado — Train: {df_train.shape}, Test: {df_test.shape}')
    return df_train, df_test

### 5.12 NLP Embedding Features

Features de análisis de texto avanzado sobre la columna `description`. Se combinan cuatro enfoques complementarios:

| Componente | Features | Descripción |
|---|---|---|
| TF-IDF + SVD (LSA) | `tfidf_svd_00..39` (40) | Temas latentes del vocabulario del listado; robusto con boilerplate |
| SBERT + PCA | `sbert_pca_00..31` (32) | Semántica profunda comprimida; `all-MiniLM-L6-v2`, CPU, 384d → 32d |
| Sentimiento VADER | `sent_neg/neu/pos/compound` (4) | Tono del texto: "needs TLC" vs "stunning ocean views" |
| Métricas lexicales | `avg_word_length`, `type_token_ratio`, `sentence_count` (3) | Profundidad y diversidad del vocabulario del vendedor |

**Total: 79 features nuevos.**

> `sentence-transformers` y `nltk` deben estar instalados:
> ```bash
> pip install sentence-transformers nltk
> python -m nltk.downloader vader_lexicon
> ```
> Todos los parámetros estadísticos (vocab, SVD, PCA) se aprenden exclusivamente de `df_train`.

In [ ]:
# ── Instalar dependencias NLP (ejecutar una sola vez) ─────────────────────────
import subprocess, sys

_nlp_packages = [
    'sentence-transformers',
    'nltk',
    'transformers',
    'torch',
    'sentencepiece',
]

for pkg in _nlp_packages:
    mod = pkg.replace('-', '_').split('[')[0]
    try:
        __import__(mod)
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} instalado.')

import nltk
nltk.download('vader_lexicon', quiet=True)
print('Dependencias NLP listas.')

In [ ]:
def add_nlp_embedding_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    svd_components: int = 40,
    pca_components: int = 32,
    sbert_model_name: str = 'all-MiniLM-L6-v2',
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features avanzados de NLP sobre la columna 'description'.

    Componentes:
        1. TF-IDF (5 000 features, unigramas + bigramas) + TruncatedSVD (LSA)
               → tfidf_svd_00 .. tfidf_svd_{svd_components-1}
        2. Sentence-BERT embeddings (CPU, normalizado) + PCA
               → sbert_pca_00 .. sbert_pca_{pca_components-1}
        3. Análisis de sentimiento VADER (sin GPU, sin entrenamiento)
               → sent_neg, sent_neu, sent_pos, sent_compound
        4. Métricas lexicales
               → avg_word_length, type_token_ratio, sentence_count

    Stats aprendidas SOLO de df_train: vocab TF-IDF, componentes SVD, PCA fit.
    VADER y métricas lexicales son deterministas sin parámetros entrenables.

    Requiere:
        pip install sentence-transformers nltk scikit-learn
        python -m nltk.downloader vader_lexicon
    """
    import nltk
    nltk.download('vader_lexicon', quiet=True)
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD, PCA
    from sentence_transformers import SentenceTransformer

    df_train = df_train.copy()
    df_test  = df_test.copy()

    train_texts = df_train['description'].fillna('').tolist()
    test_texts  = df_test['description'].fillna('').tolist()

    # ── 1. TF-IDF + TruncatedSVD (LSA) ───────────────────────────────────────
    print('  NLP: TF-IDF + TruncatedSVD...')
    tfidf = TfidfVectorizer(
        max_features=5_000,
        ngram_range=(1, 2),
        min_df=3,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'\b[a-zA-Z]{2,}\b',
    )
    svd = TruncatedSVD(n_components=svd_components, random_state=seed)

    X_tr_tfidf = tfidf.fit_transform(train_texts)
    X_te_tfidf = tfidf.transform(test_texts)
    X_tr_svd   = svd.fit_transform(X_tr_tfidf)
    X_te_svd   = svd.transform(X_te_tfidf)

    svd_cols = [f'tfidf_svd_{i:02d}' for i in range(svd_components)]
    df_train[svd_cols] = X_tr_svd.astype(np.float32)
    df_test[svd_cols]  = X_te_svd.astype(np.float32)
    print(f'    Varianza acumulada SVD: {svd.explained_variance_ratio_.sum():.1%}')

    # ── 2. Sentence-BERT + PCA ────────────────────────────────────────────────
    print(f'  NLP: SBERT {sbert_model_name!r} (CPU)...')
    sbert  = SentenceTransformer(sbert_model_name, device='cpu')
    tr_emb = sbert.encode(
        train_texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    te_emb = sbert.encode(
        test_texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    print('  NLP: PCA sobre embeddings SBERT...')
    pca      = PCA(n_components=pca_components, random_state=seed)
    X_tr_pca = pca.fit_transform(tr_emb)
    X_te_pca = pca.transform(te_emb)

    pca_cols = [f'sbert_pca_{i:02d}' for i in range(pca_components)]
    df_train[pca_cols] = X_tr_pca.astype(np.float32)
    df_test[pca_cols]  = X_te_pca.astype(np.float32)
    print(f'    Varianza acumulada PCA: {pca.explained_variance_ratio_.sum():.1%}')

    # ── 3. Análisis de sentimiento VADER ──────────────────────────────────────
    print('  NLP: Sentiment VADER...')
    sia = SentimentIntensityAnalyzer()
    for df, texts in [(df_train, train_texts), (df_test, test_texts)]:
        scores = [sia.polarity_scores(t) for t in texts]
        df['sent_neg']      = np.array([s['neg']      for s in scores], np.float32)
        df['sent_neu']      = np.array([s['neu']      for s in scores], np.float32)
        df['sent_pos']      = np.array([s['pos']      for s in scores], np.float32)
        df['sent_compound'] = np.array([s['compound'] for s in scores], np.float32)

    # ── 4. Métricas lexicales ─────────────────────────────────────────────────
    def _lex(texts):
        awl_v, ttr_v, sc_v = [], [], []
        for t in texts:
            words = t.split()
            n = len(words)
            if n == 0:
                awl_v.append(0.0); ttr_v.append(0.0); sc_v.append(0)
                continue
            awl_v.append(float(np.mean([len(w) for w in words])))
            ttr_v.append(len(set(words)) / n)
            sc_v.append(max(1, t.count('.') + t.count('!') + t.count('?')))
        return (np.array(awl_v, np.float32),
                np.array(ttr_v, np.float32),
                np.array(sc_v,  np.int32))

    for df, texts in [(df_train, train_texts), (df_test, test_texts)]:
        awl, ttr_vals, sc = _lex(texts)
        df['avg_word_length']  = awl
        df['type_token_ratio'] = ttr_vals
        df['sentence_count']   = sc

    n_new = svd_components + pca_components + 4 + 3
    print(f'  NLP embeddings completado — {n_new} features nuevos por fila.')
    return df_train, df_test

### 5.13 Price Perception Classifier — Fine-tuning OOF

Fine-tunea un transformer pequeño para clasificar el rango de precio percibido a partir del texto de la descripción, y exporta sus probabilidades como features.

| Parámetro | Descripción |
|---|---|
| `model_name` | Modelo HuggingFace base (`bert-tiny` por defecto para CPU) |
| `n_folds` | Número de folds para OOF (5) |
| `n_epochs` | Épocas de entrenamiento por fold / modelo final (3) |
| `max_length` | Longitud máxima de tokens (128 — cubre ~98 % de las descripciones) |

**Features generadas:** `nlp_price_tier_pred`, `nlp_price_tier_conf`, `nlp_price_tier_entropy`, `nlp_pt_proba_00..09` (13 features en total).

> Esta función **no** forma parte del loop `FE_STEPS` porque requiere ~20–40 min de CPU. Se ejecuta por separado en la sección 7.

In [ ]:
# prajjwal1/bert-tiny requiere sentencepiece y falla en entornos sin el — usar distilbert
_NLP_CLF_DEFAULT_LR = {
    'distilbert-base-uncased'           : 2e-5,   # recomendado CPU — ~50 min
    'bert-base-uncased'                 : 2e-5,
    'huawei-noah/TinyBERT_General_4L_312D': 3e-4, # alternativa rapida si disponible
}


def build_price_perception_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    model_name: str = 'distilbert-base-uncased',
    n_folds: int = 5,
    n_epochs: int = 3,
    batch_size: int = 16,
    max_length: int = 128,
    model_save_dir: str = '../../models/nlp_price_classifier',
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fine-tunea un clasificador de texto (description → price_bin) y agrega
    sus salidas como features.

    Estrategia anti-leakage — OOF obligatorio:
        price_bin se deriva de lastSoldPrice_hpi_adjusted (el target).
        Usar prediction(text → bin) como feature de train SIN OOF implica que
        el modelo vio el label durante su propio entrenamiento.
        Solución: StratifiedKFold(n_folds); cada fila de train se predice solo
        por modelos que nunca la vieron. El modelo final (100 % train) se usa
        únicamente sobre el test set.

    Features generadas (por fila):
        nlp_price_tier_pred     — clase predicha (int 0–9 = Q1–Q10)
        nlp_price_tier_conf     — max(softmax)  (confianza en la predicción)
        nlp_price_tier_entropy  — entropía de la distribución (ambigüedad del texto)
        nlp_pt_proba_00..09     — probabilidad completa por cuantil

    Args:
        model_name    : HuggingFace model ID. Default 'prajjwal1/bert-tiny' (CPU rápido).
                        Para mayor calidad usar 'distilbert-base-uncased'.
        model_save_dir: directorio donde se persiste el modelo final.

    df_train debe contener las columnas 'description' y 'price_bin'.
    """
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from sklearn.model_selection import StratifiedKFold

    device = torch.device('cpu')
    lr     = _NLP_CLF_DEFAULT_LR.get(model_name, 2e-5)

    df_train = df_train.copy()
    df_test  = df_test.copy()

    Path(model_save_dir).mkdir(parents=True, exist_ok=True)

    # ── Etiquetas ─────────────────────────────────────────────────────────────
    n_classes = 10
    bin_order = [f'Q{i}' for i in range(1, n_classes + 1)]
    label_map = {b: i for i, b in enumerate(bin_order)}

    raw_bins     = df_train['price_bin'].astype(str)
    all_labels   = np.array([label_map.get(b, -1) for b in raw_bins], np.int64)
    valid_mask   = all_labels >= 0
    n_invalid    = int((~valid_mask).sum())
    if n_invalid:
        print(f'  Advertencia: {n_invalid} fila(s) sin price_bin válido — excluidas del entrenamiento.')

    train_texts  = df_train['description'].fillna('').tolist()
    test_texts   = df_test['description'].fillna('').tolist()
    valid_idx    = np.where(valid_mask)[0]
    valid_labels = all_labels[valid_idx]

    # ── Tokenización ──────────────────────────────────────────────────────────
    print(f'  [NLP Clf] Tokenizando con {model_name!r} (max_length={max_length})...')
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    def _tok(texts):
        return tokenizer(texts, padding='max_length', truncation=True,
                         max_length=max_length, return_tensors='pt')

    enc_all_train = _tok(train_texts)
    enc_test      = _tok(test_texts)
    enc_valid     = {k: v[valid_idx] for k, v in enc_all_train.items()}
    has_tti       = 'token_type_ids' in enc_all_train

    # ── Helpers ───────────────────────────────────────────────────────────────
    def _build_model():
        return AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=n_classes,
        ).to(device)

    def _loader(enc, labels=None, shuffle=False):
        parts = [enc['input_ids'], enc['attention_mask']]
        if has_tti:
            parts.append(enc['token_type_ids'])
        if labels is not None:
            parts.append(torch.tensor(labels, dtype=torch.long))
        return DataLoader(TensorDataset(*parts), batch_size=batch_size, shuffle=shuffle)

    def _fwd(model, batch):
        if has_tti:
            ids, mask, tti = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            lbl = batch[3].to(device) if len(batch) > 3 else None
            return model(input_ids=ids, attention_mask=mask, token_type_ids=tti, labels=lbl)
        else:
            ids, mask = batch[0].to(device), batch[1].to(device)
            lbl = batch[2].to(device) if len(batch) > 2 else None
            return model(input_ids=ids, attention_mask=mask, labels=lbl)

    def _train_epoch(model, loader, optimizer):
        model.train()
        total = 0.0
        for batch in loader:
            optimizer.zero_grad()
            out = _fwd(model, batch)
            out.loss.backward()
            optimizer.step()
            total += out.loss.item()
        return total / max(1, len(loader))

    @torch.no_grad()
    def _predict(model, loader, n):
        model.eval()
        proba = np.zeros((n, n_classes), np.float32)
        idx = 0
        for batch in loader:
            out = _fwd(model, batch)
            p   = F.softmax(out.logits, dim=-1).cpu().numpy()
            proba[idx: idx + p.shape[0]] = p
            idx += p.shape[0]
        return proba

    # ── OOF — StratifiedKFold sobre filas de train con label válido ───────────
    print(f'\n  [NLP Clf] OOF {n_folds}-fold  ({len(valid_idx)} muestras válidas, modelo: {model_name})')
    skf       = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_proba = np.zeros((len(df_train), n_classes), np.float32)

    for fold, (tr_i, va_i) in enumerate(skf.split(valid_idx, valid_labels)):
        print(f'  Fold {fold + 1}/{n_folds}  train={len(tr_i)}  val={len(va_i)}')
        enc_f_tr = {k: v[tr_i] for k, v in enc_valid.items()}
        enc_f_va = {k: v[va_i] for k, v in enc_valid.items()}
        ldr_tr   = _loader(enc_f_tr, valid_labels[tr_i], shuffle=True)
        ldr_va   = _loader(enc_f_va, shuffle=False)

        model     = _build_model()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        for ep in range(n_epochs):
            loss = _train_epoch(model, ldr_tr, optimizer)
            print(f'    epoch {ep + 1}/{n_epochs}  loss={loss:.4f}')

        fold_proba                 = _predict(model, ldr_va, len(va_i))
        oof_proba[valid_idx[va_i]] = fold_proba
        del model

    # ── Modelo final (100 % train válido) — para predecir test ────────────────
    print(f'\n  [NLP Clf] Modelo final sobre 100 % del train ({len(valid_idx)} muestras)...')
    ldr_full    = _loader(enc_valid, valid_labels, shuffle=True)
    final_model = _build_model()
    optimizer   = torch.optim.AdamW(final_model.parameters(), lr=lr, weight_decay=0.01)
    for ep in range(n_epochs):
        loss = _train_epoch(final_model, ldr_full, optimizer)
        print(f'  epoch {ep + 1}/{n_epochs}  loss={loss:.4f}')

    final_model.save_pretrained(model_save_dir)
    tokenizer.save_pretrained(model_save_dir)
    print(f'  Modelo guardado en: {model_save_dir}')

    print('  [NLP Clf] Prediciendo sobre test...')
    ldr_te     = _loader(enc_test, shuffle=False)
    test_proba = _predict(final_model, ldr_te, len(df_test))
    del final_model

    # ── Adjuntar features ─────────────────────────────────────────────────────
    def _attach(df, proba):
        df['nlp_price_tier_pred']    = proba.argmax(1).astype(np.int32)
        df['nlp_price_tier_conf']    = proba.max(1).astype(np.float32)
        eps = 1e-9
        df['nlp_price_tier_entropy'] = (
            -(proba * np.log(proba + eps)).sum(1)
        ).astype(np.float32)
        for c in range(n_classes):
            df[f'nlp_pt_proba_{c:02d}'] = proba[:, c].astype(np.float32)

    _attach(df_train, oof_proba)
    _attach(df_test,  test_proba)

    print(f'  [NLP Clf] {3 + n_classes} features añadidos (OOF en train, modelo final → test).')
    return df_train, df_test

---
## Recomendación de Modelo de Texto

| Caso de uso | Modelo | Justificación |
|---|---|---|
| **Embeddings generales** | `sentence-transformers/all-MiniLM-L6-v2` | Mejor balance velocidad/calidad en CPU: 22 M params, salida 384d, ~5× más rápido que `all-mpnet-base-v2` con solo ~5 % menos calidad en SBERT BEIR benchmarks |
| **Embeddings alta calidad** | `sentence-transformers/all-mpnet-base-v2` | Mayor calidad semántica; recomendado si se dispone de más tiempo o GPU |
| **Fine-tuning rápido (CPU)** | `huawei-noah/TinyBERT_General_4L_312D` | 14.5 M params — ~5× más rápido que DistilBERT; usa tokenizer BERT estándar (sin sentencepiece) |
| **Fine-tuning calidad** | `distilbert-base-uncased` | 66 M params, ~40 % más rápido que BERT-base con calidad cercana; ajustar `lr=2e-5` |
| **Análisis de sentimiento** | VADER (`nltk`) | Sin GPU, sin entrenamiento, léxico ajustado para inglés informal y texto de marketing — correcto para listados inmobiliarios |

### Sobre el anti-leakage en el clasificador de precio

`price_bin` se deriva directamente de `lastSoldPrice_hpi_adjusted`. Entrenar un clasificador de texto en `(description → price_bin)` y luego usar su salida como feature del mismo set de entrenamiento **es leakage** — el modelo downstream aprendería a confiar en una predicción que ya conoce la respuesta correcta.

La solución implementada replica la estrategia de *out-of-fold stacking*:

```
Para cada fold k de StratifiedKFold(5):
    Entrenar en los otros 4 folds  (nunca ve el fold k)
    Predecir fold k  →  oof_proba[fold_k_indices]

Entrenar modelo final en 100 % del train
Predecir sobre test  →  test_proba
```

Esto garantiza que cada fila de train fue predicha por un modelo que nunca la vio, replicando exactamente las condiciones de inferencia en producción.

---
## 6. Aplicar Feature Engineering — Incremental

`FE_CONFIG` controla qué pasos se re-ejecutan en esta corrida.

| Comportamiento | Condición |
|---|---|
| **Crea** los parquet desde cero | Los archivos no existen aún |
| **Actualiza** solo las columnas del paso habilitado | Los parquet ya existen |

Esto permite modificar una función individual (p. ej. un FE con transformers costoso) y regenerar solo sus columnas sin tocar el resto.

> **Dependencias entre pasos**: `text` requiere columnas de `amenities`; `cross` requiere columnas de todos los pasos anteriores. Si se habilita un paso dependiente sin su dependencia y no hay parquet previo, la ejecución fallará con `KeyError`.

In [ ]:
# ── Qué pasos recalcular en esta corrida ──────────────────────────────────────
# True  → ejecuta la función y sobreescribe sus columnas en el parquet.
# False → deja las columnas del parquet intactas (o las omite si no existe aún).
FE_CONFIG = {
    'size'          : True,
    'rooms'         : True,
    'age'           : True,
    'financial'     : True,
    'geo'           : True,
    'schools'       : True,
    'amenities'     : True,
    'listing'       : True,
    'text'          : True,    # depende de amenities
    'cross'         : True,    # depende de todos los anteriores
    'nlp_embeddings': True,    # TF-IDF/SVD + SBERT+PCA + VADER + léxico (~5 min CPU)
}

print('FE_CONFIG:')
for step, enabled in FE_CONFIG.items():
    print(f'  {step:<16} {"ON" if enabled else "off"}')

FE_CONFIG:
  size             ON
  rooms            ON
  age              ON
  financial        ON
  geo              ON
  schools          ON
  amenities        ON
  listing          ON
  text             ON
  cross            ON
  nlp_embeddings   ON


In [ ]:
FE_STEPS = [
    ('size',            add_size_features),
    ('rooms',           add_room_features),
    ('age',             add_property_age_features),
    ('financial',       add_financial_features),
    ('geo',             add_geographic_features),
    ('schools',         add_school_features),
    ('amenities',       add_amenity_features),
    ('listing',         add_listing_features),
    ('text',            add_text_features),
    ('cross',           add_cross_features),
    ('nlp_embeddings',  add_nlp_embedding_features),
]

fe_paths      = {k: fe_output + v for k, v in fe_files.items()}
enabled_steps = [name for name, _ in FE_STEPS if FE_CONFIG.get(name, False)]
print(f'Pasos habilitados ({len(enabled_steps)}/11): {enabled_steps}')

# ── Cargar base: parquet existente o datos del split ─────────────────────────
parquets_exist = (
    os.path.exists(fe_paths['train']) and os.path.exists(fe_paths['test'])
)

if parquets_exist:
    df_tr_state = pd.read_parquet(fe_paths['train'])
    _df_val_base = pd.read_parquet(fe_paths['test'])
    if USE_META_TRAIN:
        _df_mt_base = (
            pd.read_parquet(fe_paths['meta_train'])
            if os.path.exists(fe_paths['meta_train'])
            else df_meta_train.copy().reset_index(drop=True)
        )
    print(f'  Parquet existente cargado (train {df_tr_state.shape}, val {_df_val_base.shape}).')
else:
    df_tr_state  = df_train.copy().reset_index(drop=True)
    _df_val_base = df_meta_test.copy().reset_index(drop=True)
    if USE_META_TRAIN:
        _df_mt_base = df_meta_train.copy().reset_index(drop=True)
    print('  No hay parquet previo — se crea desde cero.')

# ── Preservar zpid y target ───────────────────────────────────────────────────
zpids_train = df_tr_state['zpid'].values.copy()
y_train     = (df_tr_state[target].values.copy()
               if target in df_tr_state.columns else df_train[target].values)

zpids_val   = _df_val_base['zpid'].values.copy()
y_val       = (_df_val_base[target].values.copy()
               if target in _df_val_base.columns else df_meta_test[target].values)

if USE_META_TRAIN:
    zpids_meta = _df_mt_base['zpid'].values.copy()
    y_meta     = (_df_mt_base[target].values.copy()
                  if target in _df_mt_base.columns else df_meta_train[target].values)
    n_meta     = len(_df_mt_base)
    _unified   = pd.concat([_df_mt_base, _df_val_base], axis=0, ignore_index=True)
else:
    _unified   = _df_val_base.copy().reset_index(drop=True)

df_val_state = _unified.drop(columns=[target], errors='ignore')

# ── Aplicar pasos habilitados a train y val ───────────────────────────────────
print('\nAplicando pasos de FE a train y val:')
for name, fn in FE_STEPS:
    if not FE_CONFIG.get(name, False):
        continue
    cols_before               = set(df_tr_state.columns)
    df_tr_state, df_val_state = fn(df_tr_state, df_val_state)
    added = sorted(set(df_tr_state.columns) - cols_before)
    n_new = len(added)
    if n_new:
        print(f'  [{name:16s}] +{n_new} col(s): {added}')
    else:
        print(f'  [{name:16s}]  actualizacion')

# ── Re-split unified val ──────────────────────────────────────────────────────
if USE_META_TRAIN:
    df_mt_final  = df_val_state.iloc[:n_meta].reset_index(drop=True).copy()
    df_val_final = df_val_state.iloc[n_meta:].reset_index(drop=True).copy()
else:
    df_val_final = df_val_state.reset_index(drop=True).copy()
    df_mt_final  = None

df_tr_final = df_tr_state.copy()

# ── Restaurar zpid y target ───────────────────────────────────────────────────
df_tr_final['zpid']  = zpids_train
df_tr_final[target]  = y_train
df_val_final['zpid'] = zpids_val
df_val_final[target] = y_val
if USE_META_TRAIN:
    df_mt_final['zpid'] = zpids_meta
    df_mt_final[target] = y_meta

# ── Persistir train y val ─────────────────────────────────────────────────────
print('\nGuardando parquets train / val...')
df_tr_final.to_parquet(fe_paths['train'], index=False)
df_val_final.to_parquet(fe_paths['test'],  index=False)
print(f'  {fe_paths["train"]}  {df_tr_final.shape}')
print(f'  {fe_paths["test"]}   {df_val_final.shape}')
if USE_META_TRAIN:
    df_mt_final.to_parquet(fe_paths['meta_train'], index=False)
    print(f'  {fe_paths["meta_train"]}  {df_mt_final.shape}')

# ── Aplicar FE al competition test (test_processed.csv, sin target) ──────────
print('\nAplicando FE al competition test...')
comp_path = fe_paths['competition_test']

if os.path.exists(comp_path):
    df_comp_state = pd.read_parquet(comp_path)
    print(f'  Parquet existente cargado ({df_comp_state.shape})')
else:
    df_comp_state = df_test.copy().reset_index(drop=True)
    print('  No hay parquet previo — se crea desde cero.')

zpids_comp = df_comp_state['zpid'].values.copy()

for name, fn in FE_STEPS:
    if not FE_CONFIG.get(name, False):
        continue
    # Stats aprendidas de df_tr_final (80% de train, ya procesado)
    _, df_comp_state = fn(df_tr_final, df_comp_state)
    print(f'  [{name:16s}] OK')

df_comp_state['zpid'] = zpids_comp
df_comp_state.to_parquet(comp_path, index=False)
print(f'  {comp_path}  {df_comp_state.shape}')

# ── Alias finales ─────────────────────────────────────────────────────────────
df_train_fe            = df_tr_final
df_test_fe             = df_val_final       # 20% split — para evaluar el modelo
df_meta_train_fe       = df_mt_final
df_competition_test_fe = df_comp_state      # test_processed.csv — para predicciones finales

# ── Validar universos ─────────────────────────────────────────────────────────
assert set(df_train_fe['zpid'])            == set(df_train['zpid']),     'IDs train no coinciden'
assert set(df_test_fe['zpid'])             == set(df_meta_test['zpid']),  'IDs val no coinciden'
assert set(df_competition_test_fe['zpid']) == set(df_test['zpid']),      'IDs competition test no coinciden'
if USE_META_TRAIN:
    assert set(df_meta_train_fe['zpid']) == set(df_meta_train['zpid']), 'IDs meta_train no coinciden'

print('\nValidacion OK — universos preservados.')
print('\nShapes post-FE:')
print(f'  df_train_fe            : {df_train_fe.shape}  (80% de train_processed)')
if USE_META_TRAIN:
    print(f'  df_meta_train_fe       : {df_meta_train_fe.shape}')
print(f'  df_test_fe             : {df_test_fe.shape}  (20% de train_processed — validacion)')
print(f'  df_competition_test_fe : {df_competition_test_fe.shape}  (test_processed.csv completo)')
print(f'\nTotal features (train): {df_train_fe.shape[1] - 2}  (excl. zpid y target)')

Pasos habilitados (11/11): ['size', 'rooms', 'age', 'financial', 'geo', 'schools', 'amenities', 'listing', 'text', 'cross', 'nlp_embeddings']


  Parquet existente cargado (train (9472, 219), val (2368, 219)).

Aplicando pasos de FE a train y val:
  [size            ]  actualizacion
  [rooms           ]  actualizacion
  [age             ]  actualizacion
  [financial       ]  actualizacion
  [geo             ]  actualizacion
  [schools         ]  actualizacion
  [amenities       ]  actualizacion
  [listing         ]  actualizacion
  [text            ]  actualizacion
  [cross           ]  actualizacion
  NLP: TF-IDF + TruncatedSVD...
    Varianza acumulada SVD: 43.3%
  NLP: SBERT 'all-MiniLM-L6-v2' (CPU)...


Batches: 100%|██████████| 37/37 [00:30<00:00,  1.20it/s]


  NLP: PCA sobre embeddings SBERT...
    Varianza acumulada PCA: 63.5%
  NLP: Sentiment VADER...
  NLP embeddings completado — 79 features nuevos por fila.
  [nlp_embeddings  ]  actualizacion

Guardando parquets train / val...
  ../../data/tabular/train_fe_sent.parquet  (9472, 219)
  ../../data/tabular/test_fe_sent.parquet   (2368, 219)

Aplicando FE al competition test...
  Parquet existente cargado ((5038, 216))
  [size            ] OK
  [rooms           ] OK
  [age             ] OK
  [financial       ] OK
  [geo             ] OK
  [schools         ] OK
  [amenities       ] OK
  [listing         ] OK
  [text            ] OK
  [cross           ] OK
  NLP: TF-IDF + TruncatedSVD...
    Varianza acumulada SVD: 43.3%
  NLP: SBERT 'all-MiniLM-L6-v2' (CPU)...


Batches: 100%|██████████| 79/79 [01:07<00:00,  1.17it/s]


  NLP: PCA sobre embeddings SBERT...
    Varianza acumulada PCA: 63.5%
  NLP: Sentiment VADER...
  NLP embeddings completado — 79 features nuevos por fila.
  [nlp_embeddings  ] OK
  ../../data/tabular/competition_test_fe_sent.parquet  (5038, 216)

Validacion OK — universos preservados.

Shapes post-FE:
  df_train_fe            : (9472, 219)  (80% de train_processed)
  df_test_fe             : (2368, 219)  (20% de train_processed — validacion)
  df_competition_test_fe : (5038, 216)  (test_processed.csv completo)

Total features (train): 217  (excl. zpid y target)


---
## 7. Price Perception Classifier — Ejecución

Ejecuta el fine-tuning OOF sobre los parquets ya guardados por la sección 6.

**Tiempo estimado en CPU:**

| Modelo | Tiempo OOF (5 folds × 3 épocas) | Tiempo modelo final | Total aprox. |
|---|---|---|---|
| `distilbert-base-uncased` (default) | ~40 min | ~8 min | **~50 min** |
| `huawei-noah/TinyBERT_General_4L_312D` | ~10 min | ~2 min | **~12 min** |
| `bert-base-uncased` | ~75 min | ~15 min | **~90 min** |

> `prajjwal1/bert-tiny` requiere `sentencepiece` incluso con `use_fast=False` — no usar.

**Features resultantes (13):** `nlp_price_tier_pred`, `nlp_price_tier_conf`, `nlp_price_tier_entropy`, `nlp_pt_proba_00..09`

> Poner `NLP_CLF_CONFIG['run'] = False` para saltear el paso sin borrar las features si ya existen en el parquet.

In [ ]:
# ── Configuracion del clasificador de percepcion de precio ──────────────────
NLP_CLF_CONFIG = {
    'run'            : True,
    'model_name'     : 'distilbert-base-uncased',  # prajjwal1/bert-tiny requiere sentencepiece
    'n_folds'        : 5,
    'n_epochs'       : 3,
    'batch_size'     : 16,
    'max_length'     : 128,
    'model_save_dir' : '../../models/nlp_price_classifier',
}

# Forzar CPU: ocultar GPUs antes de cualquier import de torch/CUDA
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

# Imports pesados al inicio del cell — necesarios tanto para fine-tuning como para inferencia
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if NLP_CLF_CONFIG['run']:
    # Guard: competition_test parquet debe existir (generado en seccion 6)
    comp_path = fe_paths['competition_test']
    if not os.path.exists(comp_path):
        raise FileNotFoundError(
            f'No se encontro el parquet de competition test en:\n  {comp_path}\n'
            'Ejecuta primero la seccion 6 con el nuevo codigo (incluye competition_test).'
        )

    # ── Train (80%) y val (20%) — OOF en train, modelo final en val ──────────
    df_tr_clf  = pd.read_parquet(fe_paths['train'])
    df_val_clf = pd.read_parquet(fe_paths['test'])
    print(f'Parquets cargados — train {df_tr_clf.shape}, val {df_val_clf.shape}')

    df_tr_clf, df_val_clf = build_price_perception_features(
        df_tr_clf, df_val_clf,
        model_name     = NLP_CLF_CONFIG['model_name'],
        n_folds        = NLP_CLF_CONFIG['n_folds'],
        n_epochs       = NLP_CLF_CONFIG['n_epochs'],
        batch_size     = NLP_CLF_CONFIG['batch_size'],
        max_length     = NLP_CLF_CONFIG['max_length'],
        model_save_dir = NLP_CLF_CONFIG['model_save_dir'],
    )

    df_tr_clf.to_parquet(fe_paths['train'], index=False)
    df_val_clf.to_parquet(fe_paths['test'],  index=False)
    print(f'  train : {fe_paths["train"]}  {df_tr_clf.shape}')
    print(f'  val   : {fe_paths["test"]}   {df_val_clf.shape}')

    # ── Competition test — inferencia directa con el modelo guardado ──────────
    print('\n  [NLP Clf] Inferencia en competition test con modelo guardado...')

    df_comp_clf = pd.read_parquet(comp_path)
    comp_texts  = df_comp_clf['description'].fillna('').tolist()

    clf_tok   = AutoTokenizer.from_pretrained(NLP_CLF_CONFIG['model_save_dir'], use_fast=False)
    clf_model = AutoModelForSequenceClassification.from_pretrained(
        NLP_CLF_CONFIG['model_save_dir']
    ).eval()

    enc_comp = clf_tok(
        comp_texts, padding='max_length', truncation=True,
        max_length=NLP_CLF_CONFIG['max_length'], return_tensors='pt',
    )
    has_tti_c  = 'token_type_ids' in enc_comp
    comp_parts = [enc_comp['input_ids'], enc_comp['attention_mask']]
    if has_tti_c:
        comp_parts.append(enc_comp['token_type_ids'])
    ldr_comp = DataLoader(
        TensorDataset(*comp_parts),
        batch_size=NLP_CLF_CONFIG['batch_size'], shuffle=False,
    )

    comp_proba = np.zeros((len(df_comp_clf), 10), np.float32)
    ci = 0
    with torch.no_grad():
        for batch in ldr_comp:
            if has_tti_c:
                ids, mask, tti = batch
                out = clf_model(input_ids=ids, attention_mask=mask, token_type_ids=tti)
            else:
                ids, mask = batch
                out = clf_model(input_ids=ids, attention_mask=mask)
            p = F.softmax(out.logits, dim=-1).cpu().numpy()
            comp_proba[ci:ci + p.shape[0]] = p
            ci += p.shape[0]

    df_comp_clf['nlp_price_tier_pred']    = comp_proba.argmax(1).astype(np.int32)
    df_comp_clf['nlp_price_tier_conf']    = comp_proba.max(1).astype(np.float32)
    eps = 1e-9
    df_comp_clf['nlp_price_tier_entropy'] = (
        -(comp_proba * np.log(comp_proba + eps)).sum(1)
    ).astype(np.float32)
    for c in range(10):
        df_comp_clf[f'nlp_pt_proba_{c:02d}'] = comp_proba[:, c].astype(np.float32)

    df_comp_clf.to_parquet(comp_path, index=False)
    print(f'  comp  : {comp_path}  {df_comp_clf.shape}')

    # Alias finales para usar en celdas siguientes
    df_train_fe            = df_tr_clf
    df_test_fe             = df_val_clf
    df_competition_test_fe = df_comp_clf

    print(f'\nShapes finales:')
    print(f'  df_train_fe            : {df_train_fe.shape}')
    print(f'  df_test_fe             : {df_test_fe.shape}')
    print(f'  df_competition_test_fe : {df_competition_test_fe.shape}')
    print(f'  Total features: {df_train_fe.shape[1] - 2}  (excl. zpid y target)')
else:
    print('NLP_CLF_CONFIG["run"] = False — paso omitido.')

Parquets cargados — train (9472, 219), val (2368, 219)
  [NLP Clf] Tokenizando con 'distilbert-base-uncased' (max_length=128)...

  [NLP Clf] OOF 5-fold  (9472 muestras válidas, modelo: distilbert-base-uncased)
  Fold 1/5  train=7577  val=1895


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4348.19it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
